<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/06_Ajuste_ponderado_covarianza_y_chi2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 06 — Ajuste ponderado, matriz de covarianza y χ² reducido**Laboratorio 1 · Clase 6****Objetivos.**1. Ajustar usando **las incertezas reales de cada punto**, y entender por qué el ajuste sin pesos es   el ajuste equivocado cuando los errores son desiguales.2. Obtener las incertezas de los parámetros de la **matriz de covarianza**, y leer la correlación   entre ellos.3. Usar el **$\\chi^2$ reducido** como diagnóstico de bondad de ajuste, incluidos los dos casos   patológicos que casi nunca se enseñan.**Requisitos previos:** Colab 05.Éste es el notebook que vas a volver a abrir todo el resto del cuatrimestre.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import curve_fitfrom scipy import statsnp.random.seed(20260916)def recta(x, a, b):    return a*x + b

---## 1. Por qué ponderarEn un laboratorio real, los puntos **no** tienen todos la misma incerteza. Ejemplos típicos:- un instrumento con error relativo constante: los valores grandes tienen error absoluto mayor;- un punto medido en el límite del rango del sensor;- un punto promediado sobre 20 repeticiones y otro sobre 3.Si algunos puntos son diez veces más confiables que otros y los tratás igual, estás tirandoinformación. Cuadrados mínimos **ponderados** minimiza$$ \\chi^2 = \\sum_{i=1}^{N} \\frac{\\left[y_i - f(x_i)\\right]^2}{\\sigma_i^2} $$es decir, cada término pesa $1/\\sigma_i^2$: cuanto más confiable el punto, más manda.

In [ ]:
# --- DATOS DE EJEMPLO: mismos que Hooke, con errores individuales realistas ---masa  = np.array([0.050, 0.100, 0.150, 0.200, 0.250, 0.300, 0.350, 0.400])elong = np.array([4.05, 8.20, 12.05, 16.40, 20.15, 24.55, 28.30, 32.60])err   = np.array([0.10, 0.12, 0.15, 0.20, 0.30, 0.45, 0.65, 0.90])   # crece con la elongación# -----------------------------------------------------------------------------p_sin, c_sin = curve_fit(recta, masa, elong)                                    # sin pesosp_con, c_con = curve_fit(recta, masa, elong, sigma=err, absolute_sigma=True)    # ponderadoe_sin, e_con = np.sqrt(np.diag(c_sin)), np.sqrt(np.diag(c_con))print(f"{'':<12}{'pendiente':>22}{'ordenada':>22}")print(f"{'sin pesos':<12}{p_sin[0]:>12.3f} ± {e_sin[0]:<7.3f}{p_sin[1]:>12.3f} ± {e_sin[1]:<7.3f}")print(f"{'ponderado':<12}{p_con[0]:>12.3f} ± {e_con[0]:<7.3f}{p_con[1]:>12.3f} ± {e_con[1]:<7.3f}")print()z = abs(p_sin[0]-p_con[0]) / np.sqrt(e_sin[0]**2 + e_con[0]**2)print(f"¿Cambió la pendiente más que su propio error?  z = {z:.2f}")

### El argumento `absolute_sigma`Éste es el detalle que se olvida y arruina el resultado. `curve_fit` tiene dos modos:- **`absolute_sigma=True`** — toma tus $\\sigma_i$ como **incertezas físicas reales**. La matriz de  covarianza que devuelve refleja esas incertezas. **Es lo que corresponde en un laboratorio**,  donde las barras de error salen de la apreciación del instrumento o de la estadística de  repeticiones.- **`absolute_sigma=False`** (el valor por defecto) — toma tus $\\sigma_i$ solo como **pesos  relativos** y reescala la covarianza para que el $\\chi^2$ reducido dé exactamente 1.El segundo modo es cómodo si no tenés idea de la escala de tus errores, pero tiene una consecuenciagrave: **destruye la información de bondad de ajuste**. Si el código fuerza $\\chi^2_\\nu = 1$, ya nopodés usar $\\chi^2_\\nu$ para saber si el modelo describe los datos.> **Regla del curso: si tenés barras de error de verdad, `absolute_sigma=True`. Siempre.**

In [ ]:
_, c_rel = curve_fit(recta, masa, elong, sigma=err, absolute_sigma=False)print("errores con absolute_sigma=True :", np.sqrt(np.diag(c_con)))print("errores con absolute_sigma=False:", np.sqrt(np.diag(c_rel)))print("\nSon distintos: el segundo está reescalado para forzar chi2_red = 1.")

---## 2. La matriz de covarianza`curve_fit` devuelve `popt` (los parámetros) y `pcov` (su matriz de covarianza). Para $p$ parámetroses una matriz $p\\times p$:- la **diagonal** contiene las varianzas: $\\sigma_{a_i} = \\sqrt{\\mathrm{pcov}[i,i]}$;- los **elementos fuera de la diagonal** contienen las covarianzas, que dicen cuán correlacionados  están los parámetros entre sí.La correlación normalizada es$\\rho_{ij} = \\mathrm{pcov}[i,j] / \\sqrt{\\mathrm{pcov}[i,i]\\,\\mathrm{pcov}[j,j]}$, y va de $-1$ a $+1$.Esto importa por una razón muy concreta: si vas a **combinar** los parámetros en una cuentaposterior (por ejemplo $g = 4\\pi^2/a$, o $\\omega_0^2 = \\omega^2 + 1/\\tau^2$), propagar tratándoloscomo independientes es incorrecto cuando $|\\rho|$ es grande.

In [ ]:
def matriz_correlacion(pcov):    d = np.sqrt(np.diag(pcov))    return pcov / np.outer(d, d)print("pcov =\n", c_con)print("\nmatriz de correlación =\n", np.round(matriz_correlacion(c_con), 3))print(f"\nρ(a, b) = {matriz_correlacion(c_con)[0,1]:.3f}")

En un ajuste lineal la pendiente y la ordenada están casi siempre fuertemente anticorreladas: sisubís una, tenés que bajar la otra para seguir pasando por la nube de puntos. Un truco clásico paradescorrelacionarlas es ajustar $y = a(x - \\bar{x}) + b'$, centrando la variable independiente.> **Ejercicio 6.1.** Ajustá con `recta(x - masa.mean(), a, b)` y volvé a calcular $\\rho$. ¿Bajó?> ¿Cambió la pendiente?

---## 3. El χ² reducidoDefinimos$$ \\chi^2 = \\sum_i \\frac{[y_i - f(x_i)]^2}{\\sigma_i^2}, \\qquad   \\chi^2_\\nu = \\frac{\\chi^2}{\\nu}, \\qquad \\nu = N - p $$donde $\\nu$ son los **grados de libertad**: cantidad de datos menos cantidad de parámetrosajustados.La idea es simple y potente: si el modelo es correcto y las barras de error están bien estimadas,cada punto debería estar típicamente a una barra de error de la curva. Entonces cada término de lasuma vale aproximadamente 1, y $\\chi^2_\\nu \\approx 1$.| valor | interpretación ||---|---|| $\\chi^2_\\nu \\approx 1$ | consistente con buen modelo **y** incertezas bien estimadas || $\\chi^2_\\nu \\gg 1$ | modelo inadecuado **o** incertezas subestimadas || $\\chi^2_\\nu \\ll 1$ | incertezas **sobre**estimadas (o parámetros de más) |El tercer caso casi nunca se enseña y es muy formativo: un ajuste "demasiado bueno" no es untriunfo, es una señal de que le pusiste barras de error más grandes de lo que corresponde. Y comolas barras infladas se propagan a los parámetros, estás reportando una incerteza final falsamentegrande.**Atención al alcance:** $\\chi^2_\\nu$ solo tiene sentido si tenés incertezas $\\sigma_i$ queestimaste independientemente. No sirve inventarlas a partir del propio ajuste — eso es circular.

In [ ]:
def chi2_reducido(y, y_modelo, sigma, n_parametros, verbose=True):    """χ² reducido y su p-valor. sigma: incertezas experimentales reales."""    y, y_modelo, sigma = map(lambda v: np.asarray(v, float), (y, y_modelo, sigma))    chi2 = np.sum(((y - y_modelo) / sigma)**2)    nu = len(y) - n_parametros    chi2r = chi2 / nu    p = stats.chi2.sf(chi2, nu)      # probabilidad de obtener un χ² igual o peor    if verbose:        if chi2r > 2:      lect = "modelo inadecuado o incertezas subestimadas"        elif chi2r < 0.5:  lect = "incertezas probablemente sobreestimadas"        else:              lect = "consistente con un buen ajuste"        print(f"χ²  = {chi2:.2f}   ν = {nu}   χ²_ν = {chi2r:.2f}   p = {p:.3f}")        print(f"  -> {lect}")    return chi2r, pchi2_reducido(elong, recta(masa, *p_con), err, n_parametros=2)

El **p-valor** es la probabilidad de obtener, por puro azar, un $\\chi^2$ igual o peor que elobservado, si el modelo fuese correcto. Un $p$ muy chico (digamos $< 0{,}01$) dice que el desacuerdodifícilmente sea casualidad. Un $p$ muy cercano a 1 es la contraparte del $\\chi^2_\\nu \\ll 1$:sospechosamente bueno.

### Los tres escenarios, lado a ladoGeneramos datos de una recta con incerteza verdadera $\\sigma = 0{,}5$ y los ajustamos declarandotres incertezas distintas.

In [ ]:
xg = np.linspace(0, 10, 15)sigma_verdadero = 0.5yg = 2.0*xg + 1.0 + np.random.normal(0, sigma_verdadero, len(xg))casos = [("σ declarado correcto (0,50)", np.full_like(xg, 0.50)),         ("σ subestimado (0,15)",        np.full_like(xg, 0.15)),         ("σ sobreestimado (2,00)",      np.full_like(xg, 2.00))]fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)for ax, (nombre, sg) in zip(axes, casos):    pp, cc = curve_fit(recta, xg, yg, sigma=sg, absolute_sigma=True)    c2r, pv = chi2_reducido(yg, recta(xg, *pp), sg, 2, verbose=False)    ax.errorbar(xg, yg, yerr=sg, fmt='o', ms=4, capsize=3)    ax.plot(xg, recta(xg, *pp), 'crimson', lw=1.6)    ax.set_title(f'{nombre}\n$\\chi^2_\\nu$ = {c2r:.2f}   (p = {pv:.3f})', fontsize=10)    ax.set_xlabel('x'); ax.grid(alpha=0.3)    print(f"{nombre:<32} a = {pp[0]:.3f} ± {np.sqrt(cc[0,0]):.3f}")axes[0].set_ylabel('y')fig.tight_layout(); plt.show()

Los tres ajustes dan **la misma pendiente** —los pesos son uniformes, así que el mínimo es el mismo—pero **incertezas del parámetro completamente distintas** y $\\chi^2_\\nu$ que delatan el problema.Es un buen recordatorio de que la incerteza que reportás no sale del ajuste: sale de tu honestidadal declarar las barras de error.

---## 4. Promedio ponderadoCaso particular útil: combinar $N$ mediciones independientes de la misma magnitud, cada una con suincerteza. La combinación óptima pesa por $1/\\sigma_i^2$:$$ \\bar{x}_p = \\frac{\\sum x_i/\\sigma_i^2}{\\sum 1/\\sigma_i^2}, \\qquad   \\sigma_{\\bar{x}_p} = \\left(\\sum \\frac{1}{\\sigma_i^2}\\right)^{-1/2} $$

In [ ]:
def promedio_ponderado(x, sigma):    x, sigma = np.asarray(x, float), np.asarray(sigma, float)    w = 1/sigma**2    xp = np.sum(w*x)/np.sum(w)    sp = 1/np.sqrt(np.sum(w))    # consistencia interna: ¿son compatibles entre sí las mediciones combinadas?    chi2 = np.sum(w*(x - xp)**2)    return xp, sp, chi2/(len(x)-1)xs = np.array([10.0, 10.03])ss = np.array([0.5, 0.02])xp, sp, c2r = promedio_ponderado(xs, ss)print(f"promedio simple    : {xs.mean():.4f}")print(f"promedio ponderado : {xp:.4f} ± {sp:.4f}   (χ²_ν de consistencia = {c2r:.2f})")

El promedio simple da 10,015, a mitad de camino entre las dos. El ponderado da prácticamente 10,03:la medición de $\\pm 0{,}02$ pesa 625 veces más que la de $\\pm 0{,}5$. Promediar sin pesos habríadegradado un resultado bueno mezclándolo con uno malo — que era la trampa del Ejercicio 4.5.

---## 5. Ejercicios**6.2.** Con tus datos del resorte y tus barras de error reales, hacé el ajuste ponderado. Reportá$k$ con su incerteza y el $\\chi^2_\\nu$. Si te da mayor que 3, no lo escondas: preguntate si elmodelo es incompleto o si subestimaste el error de lectura.**6.3.** Multiplicá todas tus barras de error por 2 y volvé a ajustar. ¿Cambia $k$? ¿Cambia$\\sigma_k$? ¿Cambia $\\chi^2_\\nu$? Explicá cada respuesta.**6.4.** Tomá el conjunto III del cuarteto de Anscombe (Colab 05), asignale barras de error de 0,5 atodos los puntos y calculá $\\chi^2_\\nu$. ¿Detecta el problema que $R$ no detectaba?**6.5.** *(criterio)* Un ajuste da $\\chi^2_\\nu = 0{,}08$ con $\\nu = 12$. El estudiante concluye queel modelo es excelente. ¿Qué le contestarías?